# Usage | 2. Network growth
This notebook explains how growbikenet orders the edges and why this is important.  
Parameters covered: `ranking`, `allow_edge_overlaps`

We start every usage notebook with the standard way of importing growbikenet:

In [ ]:
import growbikenet as gbn

We are going to work with Paris. Downloading its street network for further use:

In [ ]:
import osmnx as ox
g = ox.graph_from_place("Paris", network_type='all_public')
ox.io.save_graph_geopackage(g.to_undirected(), "Paris_streets.gpkg")

## Ranking of edges

Growbikenet not only creates a potential bicycle network, but also a ranking of the edges. The ranking metric is controlled by the parameter `ranking` which is by default set to `'betweenness_centrality'`. Other options are `'closeness_centrality'` and `'random'`. 

We are going to generate all different rankings (saved in the dictionary `edges_ranked_all`) and visualize them interactively in the end.

In [ ]:
edges_ranked_all = {}

### Betweenness centrality

Betweenness centrality is a network centrality measure approximating flow. By ordering edges like this, the first built edge is the one with highest expected flow of cyclists.

In [ ]:
edges_ranked = gbn.growbikenet("Paris",
                               ranking="betweenness_centrality",
                               street_network_file="Paris_streets.gpkg",)
edges_ranked_all["betweenness_centrality"] = edges_ranked

This ranking can be visualized using an interactive widget with a slider:

### Closeness centrality

Ranking by closeness centrality means growing from the center.

In [ ]:
edges_ranked = gbn.growbikenet("Paris",
                               ranking="closeness_centrality",
                               street_network_file="Paris_streets.gpkg",)
edges_ranked_all["closeness_centrality"] = edges_ranked

### Random

Growbikenet can also showcase random growth:

In [ ]:
edges_ranked = gbn.growbikenet("Paris",
                               ranking="random",
                               street_network_file="Paris_streets.gpkg",)
edges_ranked_all["random"] = edges_ranked

### Visualization

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
step = widgets.IntSlider(
    value=4, min=0, max=max(edges_ranked["rank"]), step=1, description="Growth step:", layout=widgets.Layout(width='500px')
)
ranking = widgets.ToggleButtons(
    options=['betweenness_centrality', 'closeness_centrality', 'random'],
    description='Ranking:',
)
def update_map(step=step, ranking=ranking):
    fig, ax = plt.subplots(figsize=(8, 6))
    edges_ranked_all[ranking][edges_ranked_all[ranking]["rank"] <= step].plot(
        linewidth=3,
        color="#096a51",
        ax=ax,
    )
    ax.set_title(f"Paris bike net growth, growth step {step}", fontsize=12)
    ax.set_ylim(edges_ranked.total_bounds[1], edges_ranked.total_bounds[3])
    ax.set_xlim(edges_ranked.total_bounds[0], edges_ranked.total_bounds[2])
    plt.axis('off')
    plt.show()

widgets.interactive(update_map, step=step, ranking=ranking)

## Allowing edge overlaps

Considerable computing time is spent on "Removing edge overlaps". 

*What are edge overlaps?* In the abstract, unrouted network, there can be no edge overlaps in the seed network. However, when the edges are routed, there are often many such overlaps. In general, we do not want to have overlaps as they make length calculations wrong, i.e., the `length` and `length_cumulative` columns in the result:

In [ ]:
edges_ranked_all["betweenness_centrality"].head()

Because edge overlaps are removed by default, the lengths are correct. In particular, the total length of all edges is

In [ ]:
int(edges_ranked_all["betweenness_centrality"].length_cumulative.iloc[-1]/1000)

kilometers.

However, if we allow edge overlaps, via `allow_edge_overlaps=True`, this will make the calculation faster:

In [ ]:
edges_ranked = gbn.growbikenet("Paris",
                               ranking="betweenness_centrality",
                               allow_edge_overlaps=True,
                               street_network_file="Paris_streets.gpkg",)

But it will also wrongly skew the calculated total length of the network to be much longer:

In [ ]:
int(edges_ranked.length_cumulative.iloc[-1]/1000)